In [1]:
%pip install --no-cache-dir --upgrade "transformers>=4.57.0"
%pip install --no-cache-dir --upgrade "tokenizers>=0.22.0"
%pip install --no-cache-dir --upgrade "safetensors>=0.6.2"
%pip install --no-cache-dir --upgrade "huggingface-hub>=0.35.0"
%pip install --no-cache-dir --upgrade "datasets>=4.0.0"
%pip install --no-cache-dir --upgrade "accelerate>=1.10.0"
%pip install --no-cache-dir --upgrade "trl>=0.29.0"
%pip install --no-cache-dir --upgrade "peft>=0.19.0"
%pip install --no-cache-dir --upgrade "sentencepiece"
%pip install --no-cache-dir --upgrade "protobuf"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 42.1 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 76.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 74.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 152.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.2/801.2 kB 207.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [transformers] [transformers]ub]
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 20.6 MB/s  0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.9.0 requires tokenizers

# 1. 데이터 전처리

In [1]:
import torch
import transformers

print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version:", torch.version.cuda)
print("device name:", torch.cuda.get_device_name(0))
print("capability:", torch.cuda.get_device_capability(0))
print("transformers version:", transformers.__version__)

torch version: 2.8.0+cu128
cuda available: True
cuda version: 12.8
device name: NVIDIA A100-SXM4-80GB
capability: (8, 0)
transformers version: 5.9.0


In [2]:
from datasets import load_dataset, Dataset
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import LoraConfig, AutoPeftModelForCausalLM
from trl import SFTConfig, SFTTrainer

In [3]:
# 1. 허깅페이스 허브에서 데이터셋 로드
dataset = load_dataset("iamjoon/finance_news_summarizer", split="train")

# 2. system_message 정의
# 데이터셋에 이미 포함된 system_prompt 열을 사용할 것이므로 따로 정의하지 않음

# 3. 원본 데이터의 type별 분포 출력
# 데이터셋에 type 열이 없으므로 전체 데이터 크기만 출력
print("전체 데이터 크기:", len(dataset))

# 4. train/test 분할 비율 설정 (0.5면 5:5로 분할)
test_ratio = 0.5

train_data = []
test_data = []

# 5. 전체 데이터의 인덱스를 train/test로 분할
data_indices = list(range(len(dataset)))
test_size = int(len(data_indices) * test_ratio)

test_data = data_indices[:test_size]
train_data = data_indices[test_size:]

# 6. OpenAI format으로 데이터 변환을 위한 함수
def format_data(sample):
    return {
        "messages": [
            {
                "role": "system",
                "content": sample["system_prompt"],
            },
            {
                "role": "user",
                "content": sample["user_prompt"],
            },
            {
                "role": "assistant",
                "content": str(sample["assistant"])
            },
        ],
    }

# 7. 분할된 데이터를 OpenAI format으로 변환
train_dataset = [format_data(dataset[i]) for i in train_data]
test_dataset = [format_data(dataset[i]) for i in test_data]

# 8. 최종 데이터셋 크기 출력
print(f"\n전체 데이터 분할 결과: Train {len(train_dataset)}개, Test {len(test_dataset)}개")

전체 데이터 크기: 991

전체 데이터 분할 결과: Train 496개, Test 495개


In [4]:
train_dataset[345]["messages"]

[{'role': 'system',
  'content': '당신은 주어진 뉴스로부터 종목에 영향을 주는 뉴스인지 판별하는 금융 뉴스 판별기입니다.\n두 가지 답변 케이스가 존재하며 무조건 파이썬의 dictionary 형식으로 작성하십시오.\n큰 따옴표 사이에 다른 따옴표들을 적으려고 시도하지 마십시오. 이는 dictionary 파싱을 실패하게 하는 원인이 됩니다. 따라서 주의하십시오.\n아래 dictionary에서 각 value는 지시사항에 해당합니다. 지사사항을 따라 적지마십시오. 해당 지시사항에 따라 적절한 value를 채워넣으십시오.\n해당사항이 없다면 빈 문자열 또는 빈 리스트로 적어야 합니다. 임의로 \'없음\' 등을 적어서는 안 됩니다.\n\n만약 해당 뉴스가 특정 종목(회사)이 언급되지 않거나, 특정 종목(회사)와 아무런 연관이 없는 뉴스일 경우에는 아래와 같이 작성합니다.\n\n답변:\n{"is_stock_related": False,\n"summary": "여기에는 해당 뉴스를 요약해서 요약문을 작성하십시오"}\n\n만약 해당 뉴스가 특정 종목(회사)들과 연관되었거나, 특정 종목(회사)과 아무런 연관이 없는 뉴스일 경우에는 아래와 같이 작성합니다.\n\n답변:\n{"is_stock_related": True,\n"positive_impact_stocks": ["파이썬 문자열 리스트의 형태로 이 뉴스가 긍정적인 영향을 줄것으로 추정되는 종목들의 이름을 작성하십시오. 약자로 적거나 별명으로 적지마십시오. 종목명으로 추정되는 한글명을 적으십시오. 뉴스로부터 추정할 수 있는 정확한 풀네임으로 적으십시오. 만약, 존재하지 않는다면 빈 리스트로 작성하십시오."],\n"reason_for_positive_impact": "위의 종목들이 해당 뉴스로부터 긍정적인 영향을 받을 것으로 추정한 이유를 여기에다가 작성하십시오",\n"positive_keywords": ["긍정적인 영향을 줄 것으로 추정되는 종목들이 존재했다면 여기에 긍정적인 영향을 주는데 근거가 

In [5]:
# 리스트 형태에서 다시 Dataset 객체로 변경
print(type(train_dataset))
print(type(test_dataset))
train_dataset = Dataset.from_list(train_dataset)
test_dataset = Dataset.from_list(test_dataset)
print(type(train_dataset))
print(type(test_dataset))

<class 'list'>
<class 'list'>
<class 'datasets.arrow_dataset.Dataset'>
<class 'datasets.arrow_dataset.Dataset'>


In [6]:
train_dataset[0]

{'messages': [{'role': 'system',
   'content': '당신은 주어진 뉴스로부터 종목에 영향을 주는 뉴스인지 판별하는 금융 뉴스 판별기입니다.\n두 가지 답변 케이스가 존재하며 무조건 파이썬의 dictionary 형식으로 작성하십시오.\n큰 따옴표 사이에 다른 따옴표들을 적으려고 시도하지 마십시오. 이는 dictionary 파싱을 실패하게 하는 원인이 됩니다. 따라서 주의하십시오.\n아래 dictionary에서 각 value는 지시사항에 해당합니다. 지사사항을 따라 적지마십시오. 해당 지시사항에 따라 적절한 value를 채워넣으십시오.\n해당사항이 없다면 빈 문자열 또는 빈 리스트로 적어야 합니다. 임의로 \'없음\' 등을 적어서는 안 됩니다.\n\n만약 해당 뉴스가 특정 종목(회사)이 언급되지 않거나, 특정 종목(회사)와 아무런 연관이 없는 뉴스일 경우에는 아래와 같이 작성합니다.\n\n답변:\n{"is_stock_related": False,\n"summary": "여기에는 해당 뉴스를 요약해서 요약문을 작성하십시오"}\n\n만약 해당 뉴스가 특정 종목(회사)들과 연관되었거나, 특정 종목(회사)과 아무런 연관이 없는 뉴스일 경우에는 아래와 같이 작성합니다.\n\n답변:\n{"is_stock_related": True,\n"positive_impact_stocks": ["파이썬 문자열 리스트의 형태로 이 뉴스가 긍정적인 영향을 줄것으로 추정되는 종목들의 이름을 작성하십시오. 약자로 적거나 별명으로 적지마십시오. 종목명으로 추정되는 한글명을 적으십시오. 뉴스로부터 추정할 수 있는 정확한 풀네임으로 적으십시오. 만약, 존재하지 않는다면 빈 리스트로 작성하십시오."],\n"reason_for_positive_impact": "위의 종목들이 해당 뉴스로부터 긍정적인 영향을 받을 것으로 추정한 이유를 여기에다가 작성하십시오",\n"positive_keywords": ["긍정적인 영향을 줄 것으로 추정되는 종목들이 존재했다면 여기에 긍정적

# 2. 모델 로드 및 템플릿 적용

In [7]:
# 허깅페이스 모델 ID
model_id = "google/gemma-4-E4B-it"

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

In [8]:
# 템플릿 적용
text = tokenizer.apply_chat_template(
    train_dataset[0]["messages"],
    tokenize=False,
    add_generation_prompt=False
)
print(text)

<bos><|turn>system
당신은 주어진 뉴스로부터 종목에 영향을 주는 뉴스인지 판별하는 금융 뉴스 판별기입니다.
두 가지 답변 케이스가 존재하며 무조건 파이썬의 dictionary 형식으로 작성하십시오.
큰 따옴표 사이에 다른 따옴표들을 적으려고 시도하지 마십시오. 이는 dictionary 파싱을 실패하게 하는 원인이 됩니다. 따라서 주의하십시오.
아래 dictionary에서 각 value는 지시사항에 해당합니다. 지사사항을 따라 적지마십시오. 해당 지시사항에 따라 적절한 value를 채워넣으십시오.
해당사항이 없다면 빈 문자열 또는 빈 리스트로 적어야 합니다. 임의로 '없음' 등을 적어서는 안 됩니다.

만약 해당 뉴스가 특정 종목(회사)이 언급되지 않거나, 특정 종목(회사)와 아무런 연관이 없는 뉴스일 경우에는 아래와 같이 작성합니다.

답변:
{"is_stock_related": False,
"summary": "여기에는 해당 뉴스를 요약해서 요약문을 작성하십시오"}

만약 해당 뉴스가 특정 종목(회사)들과 연관되었거나, 특정 종목(회사)과 아무런 연관이 없는 뉴스일 경우에는 아래와 같이 작성합니다.

답변:
{"is_stock_related": True,
"positive_impact_stocks": ["파이썬 문자열 리스트의 형태로 이 뉴스가 긍정적인 영향을 줄것으로 추정되는 종목들의 이름을 작성하십시오. 약자로 적거나 별명으로 적지마십시오. 종목명으로 추정되는 한글명을 적으십시오. 뉴스로부터 추정할 수 있는 정확한 풀네임으로 적으십시오. 만약, 존재하지 않는다면 빈 리스트로 작성하십시오."],
"reason_for_positive_impact": "위의 종목들이 해당 뉴스로부터 긍정적인 영향을 받을 것으로 추정한 이유를 여기에다가 작성하십시오",
"positive_keywords": ["긍정적인 영향을 줄 것으로 추정되는 종목들이 존재했다면 여기에 긍정적인 영향을 주는데 근거가 되었던 주요한 명사 키워드들을 파이썬 문자열 리스트 형태로 작성

Gemma 4 E4B-it 모델을 사용하므로 모델 ID는 `google/gemma-4-E4B-it`로 설정합니다.

이 예제는 뉴스 요약 텍스트 파인튜닝이므로 `AutoModelForCausalLM`과 `AutoTokenizer`를 사용합니다.

`AutoModelForCausalLM.from_pretrained()`는 사전 학습된 causal language model을 불러옵니다. 이 모델은 입력 토큰 뒤에 이어질 다음 토큰을 생성하는 방식으로 동작하므로, 뉴스 요약처럼 user 입력 뒤에 assistant 응답을 생성하는 학습에 사용할 수 있습니다.

`device_map="auto"`는 사용 가능한 GPU 장치에 모델을 자동으로 배치합니다. GPU가 한 개면 한 개에 배치되고, 여러 개면 accelerate가 가능한 범위에서 자동으로 배치합니다.

`dtype=torch.bfloat16`은 모델 가중치를 bfloat16 정밀도로 로드하도록 설정합니다. A100 GPU는 bfloat16 연산을 지원하므로 이 설정을 사용할 수 있습니다.

`AutoTokenizer.from_pretrained()`는 모델에 대응되는 토크나이저를 불러옵니다. 토크나이저는 텍스트를 모델이 처리할 수 있는 정수 토큰 시퀀스로 변환하고, 반대로 모델이 생성한 토큰을 다시 텍스트로 변환합니다.

`tokenizer.apply_chat_template()`는 `system`, `user`, `assistant`로 구성된 messages 데이터를 Gemma 4의 채팅 템플릿에 맞는 문자열로 변환합니다.

# 3. LoRA와 SFTConfig 설정

In [9]:
peft_config = LoraConfig(
    lora_alpha=32,
    lora_dropout=0.1,
    r=8,
    bias="none",
    target_modules=r".*language_model\..*\.(q_proj|v_proj)",
    task_type="CAUSAL_LM",
)

`lora_alpha`: LoRA(Low-Rank Adaptation)에서 사용하는 스케일링 계수를 설정합니다. LoRA의 가중치 업데이트가 모델에 미치는 영향을 조정하는 역할을 하며, 일반적으로 학습 안정성과 관련이 있습니다.

`lora_dropout`: LoRA 적용 시 드롭아웃 확률을 설정합니다. 드롭아웃은 과적합(overfitting)을 방지하기 위해 일부 뉴런을 랜덤하게 비활성화하는 정규화 기법입니다. `0.1`로 설정하면 학습 중 10%의 뉴런이 비활성화됩니다.

`r`: LoRA의 랭크(rank)를 설정합니다. 이는 LoRA가 학습할 저차원 공간의 크기를 결정합니다. 작은 값일수록 계산 및 메모리 효율이 높아지지만 모델의 학습 능력이 제한될 수 있습니다.

`bias`: LoRA 적용 시 편향(bias) 처리 방식을 지정합니다. `"none"`으로 설정하면 편향이 LoRA에 의해 조정되지 않습니다. `"all"` 또는 `"lora_only"`와 같은 값으로 변경하여 편향을 조정할 수도 있습니다.

`target_modules`: LoRA를 적용할 특정 모듈(레이어)의 이름을 리스트로 지정합니다. 예제에서는 `"q_proj"`와 `"v_proj"`를 지정하여, 주로 Self-Attention 메커니즘의 쿼리와 값 프로젝션 부분에 LoRA를 적용합니다.

`task_type`: LoRA가 적용되는 작업 유형을 지정합니다. `"CAUSAL_LM"`은 Causal Language Modeling, 즉 시퀀스 생성 작업에 해당합니다. 다른 예로는 `"SEQ2SEQ_LM"`(시퀀스-투-시퀀스 언어 모델링) 등이 있습니다.

In [10]:
# 최대 길이
max_seq_length=16384

In [11]:
args = SFTConfig(
    output_dir="gemma4-e4b-finance-new-summarizer",           # 저장될 디렉토리와 저장소 ID
    num_train_epochs=3,                      # 학습할 총 에포크 수 
    per_device_train_batch_size=2,           # GPU당 배치 크기
    gradient_accumulation_steps=4,           # 그래디언트 누적 스텝 수
    gradient_checkpointing=True,             # 메모리 절약을 위한 체크포인팅
    optim="adamw_torch_fused",               # 최적화기
    logging_steps=10,                        # 로그 기록 주기
    save_strategy="steps",                   # 저장 전략
    save_steps=50,                           # 저장 주기
    bf16=True,                               # bfloat16 사용
    learning_rate=1e-4,                      # 학습률
    max_grad_norm=0.3,                       # 그래디언트 클리핑
    warmup_ratio=0.03,                       # 워밍업 비율
    lr_scheduler_type="constant",            # 고정 학습률
    push_to_hub=False,                       # 허브 업로드 안 함
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},
    report_to=[],
    max_length=max_seq_length,               # 최대 시퀀스 길이 추가
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


`output_dir`: 학습 결과가 저장될 디렉토리 또는 모델 저장소의 이름을 지정합니다. 이 디렉토리에 학습된 모델 가중치, 설정 파일, 로그 파일 등이 저장됩니다.

`num_train_epochs`: 모델을 학습시키는 총 에포크(epoch) 수를 지정합니다. 에포크는 학습 데이터 전체를 한 번 순회한 주기를 의미합니다. 예를 들어, `3`으로 설정하면 데이터셋을 3번 학습합니다.

`per_device_train_batch_size`: GPU 한 대당 사용되는 배치(batch)의 크기를 설정합니다. 배치 크기는 모델이 한 번에 처리하는 데이터 샘플의 수를 의미합니다. 작은 크기는 메모리 사용량이 적지만 학습 시간이 증가할 수 있습니다.

`gradient_accumulation_steps`: 그래디언트를 누적할 스텝(step) 수를 지정합니다. 이 값이 2로 설정된 경우, 두 스텝마다 그래디언트를 업데이트합니다. 배치 크기를 가상으로 늘리는 효과가 있으며, GPU 메모리 부족 문제를 해결할 때 유용합니다.

`gradient_checkpointing`: 그래디언트 체크포인팅을 활성화하여 메모리를 절약합니다. 이 옵션은 계산 그래프를 일부 저장하지 않고 다시 계산하여 메모리를 절약하지만, 속도가 약간 느려질 수 있습니다.

`optim`: 학습 시 사용할 최적화 알고리즘을 설정합니다. `adamw_torch_fused`는 PyTorch의 효율적인 AdamW 최적화기를 사용합니다.

`logging_steps`: 로그를 기록하는 주기를 스텝 단위로 지정합니다. 예를 들어, `10`으로 설정하면 매 10 스텝마다 로그를 기록합니다.

`save_strategy`: 모델을 저장하는 전략을 설정합니다. `"steps"`로 설정된 경우, 지정된 스텝마다 모델이 저장됩니다.

`save_steps`: 모델을 저장하는 주기를 스텝 단위로 설정합니다. 예를 들어, `50`으로 설정하면 매 50 스텝마다 모델을 저장합니다.

`bf16`: bfloat16 정밀도를 사용하도록 설정합니다. bfloat16은 FP32와 유사한 범위를 제공하면서 메모리와 계산 효율성을 높입니다.

`learning_rate`: 학습률을 지정합니다. 학습률은 모델의 가중치가 한 번의 업데이트에서 얼마나 크게 변할지를 결정합니다. 일반적으로 작은 값을 사용하여 안정적인 학습을 유도합니다.

`max_grad_norm`: 그래디언트 클리핑의 임계값을 설정합니다. 이 값보다 큰 그래디언트가 발생하면, 임계값으로 조정하여 폭발적 그래디언트를 방지합니다.

`warmup_ratio`: 학습 초기 단계에서 학습률을 선형으로 증가시키는 워밍업 비율을 지정합니다. 학습의 안정성을 높이기 위해 사용됩니다.

`lr_scheduler_type`: 학습률 스케줄러의 유형을 설정합니다. `"constant"`는 학습률을 일정하게 유지합니다.

`push_to_hub`: 학습된 모델을 허브에 업로드할지 여부를 설정합니다. `False`로 설정하면 업로드하지 않습니다.

`remove_unused_columns`: 사용되지 않는 열을 제거할지 여부를 설정합니다. True로 설정하면 메모리를 절약할 수 있습니다.

`dataset_kwargs`: 데이터셋 로딩 시 추가적인 설정을 전달합니다. 예제에서는 `skip_prepare_dataset: True`로 설정하여 데이터셋 준비 단계를 건너뜁니다.

`report_to`: 학습 로그를 보고할 대상을 지정합니다. []으로 설정되면 로그가 기록되지 않습니다.

# 4. 학습 중 전처리 함수: collate_fn

In [12]:
def collate_fn(batch):
    new_batch = {
        "input_ids": [],
        "attention_mask": [],
        "labels": []
    }
    
    for example in batch:
        # messages의 각 내용에서 개행문자 제거
        clean_messages = []
        for message in example["messages"]:
            clean_message = {
                "role": message["role"],
                "content": message["content"]
            }
            clean_messages.append(clean_message)
        
        # 깨끗해진 메시지로 템플릿 적용
        text = tokenizer.apply_chat_template(
            clean_messages,
            tokenize=False,
            add_generation_prompt=False
        ).strip()
        
        # 텍스트를 토큰화
        tokenized = tokenizer(
            text,
            truncation=True,
            max_length=max_seq_length,
            padding=False,
            return_tensors=None,
        )
        
        input_ids = tokenized["input_ids"]
        attention_mask = tokenized["attention_mask"]
        
        # 레이블 초기화
        labels = [-100] * len(input_ids)
        
        prompt_text = tokenizer.apply_chat_template(
            clean_messages[:-1],
            tokenize=False,
            add_generation_prompt=True
        ).strip()
        
        prompt_tokenized = tokenizer(
            prompt_text,
            truncation=True,
            max_length=max_seq_length,
            padding=False,
            return_tensors=None,
        )
        
        assistant_start = len(prompt_tokenized["input_ids"])
        
        for j in range(assistant_start, len(input_ids)):
            labels[j] = input_ids[j]
        
        new_batch["input_ids"].append(input_ids)
        new_batch["attention_mask"].append(attention_mask)
        new_batch["labels"].append(labels)
    
    # 패딩 적용
    max_length = max(len(ids) for ids in new_batch["input_ids"])
    
    for i in range(len(new_batch["input_ids"])):
        padding_length = max_length - len(new_batch["input_ids"][i])
        
        new_batch["input_ids"][i].extend([tokenizer.pad_token_id] * padding_length)
        new_batch["attention_mask"][i].extend([0] * padding_length)
        new_batch["labels"][i].extend([-100] * padding_length)
    
    # 텐서로 변환
    for k, v in new_batch.items():
        new_batch[k] = torch.tensor(v)
    
    return new_batch

In [13]:
# collate_fn 테스트 (배치 크기 1로)
example = train_dataset[0]
batch = collate_fn([example])

print("\n처리된 배치 데이터:")
print("입력 ID 형태:", batch["input_ids"].shape)
print("어텐션 마스크 형태:", batch["attention_mask"].shape)
print("레이블 형태:", batch["labels"].shape)


처리된 배치 데이터:
입력 ID 형태: torch.Size([1, 2647])
어텐션 마스크 형태: torch.Size([1, 2647])
레이블 형태: torch.Size([1, 2647])


In [14]:
print('입력에 대한 정수 인코딩 결과:')
print(batch["input_ids"][0].tolist())

입력에 대한 정수 인코딩 결과:
[2, 105, 9731, 107, 238749, 238144, 237456, 173875, 99402, 117329, 26691, 41359, 239136, 237223, 157859, 129583, 234416, 93860, 52832, 239515, 11973, 145324, 234416, 52832, 239515, 237351, 15245, 236761, 107, 238689, 35824, 231757, 96122, 55705, 237272, 87961, 58827, 21512, 144063, 26412, 237077, 245070, 237281, 19086, 184412, 7246, 107854, 201903, 236761, 107, 240619, 27235, 244356, 238987, 180206, 44695, 27235, 244356, 238987, 43093, 31781, 237651, 81925, 9420, 237490, 37647, 17814, 111462, 236761, 117110, 19086, 26412, 241959, 237293, 183845, 26216, 31583, 22299, 91097, 76969, 236761, 116045, 158846, 201903, 236761, 107, 237534, 238528, 19086, 7726, 38541, 1550, 237170, 10384, 237462, 187222, 237223, 69471, 19773, 236761, 10384, 237470, 187222, 237293, 42813, 31781, 237308, 238098, 111462, 236761, 69471, 10384, 237462, 187222, 237223, 42813, 31781, 239498, 237384, 1550, 237482, 57206, 239592, 241376, 237651, 111462, 236761, 107, 237578, 238749, 187222, 237077, 8797

In [15]:
print('레이블에 대한 정수 인코딩 결과:')
print(batch["labels"][0].tolist())

레이블에 대한 정수 인코딩 결과:
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -1

In [16]:
# -100이 아닌 부분만 골라 디코딩
label_ids = [token_id for token_id in batch["labels"][0].tolist() if token_id != -100]

decoded_labels = tokenizer.decode(
    label_ids,
    skip_special_tokens=False,
    clean_up_tokenization_spaces=False
)

print("\nlabels 디코딩 결과 (-100 제외):")
print(decoded_labels)


labels 디코딩 결과 (-100 제외):

{'is_stock_related': True, 'negative_impact_stocks': ['여신전문금융회사', '카드사'], 'negative_keywords': ['유동성 리스크', '리볼빙', '고위험 자산', '여신전문금융회사'], 'positive_impact_stocks': [], 'positive_keywords': [], 'reason_for_negative_impact': '금융감독원장이 유동성 리스크 관리를 강조하며 무리한 영업 자제와 리볼빙 관리 강화를 지시한 것은 여신전문금융회사와 카드사들에게 부정적인 영향을 미칠 수 있습니다. 특히, 고위험 자산 확대 및 무리한 영업 확장 자제가 요구되면서 수익성에 부정적인 영향을 줄 수 있습니다.', 'reason_for_positive_impact': '', 'summary': '금융감독원장이 카드사와 여신전문금융회사를 대상으로 무리한 영업 자제와 리볼빙 관리를 당부하며, 유동성 리스크와 취약차주 대출에 대한 주의를 강조했다. 이는 해당 금융사들의 수익성에 부정적인 영향을 미칠 수 있다.'}<turn|>


## input_ids와 labels는 어떻게 생성되는가?

Gemma 4 모델의 챗 템플릿을 기준으로 input_ids와 labels 생성을 설명하겠습니다.

예를 들어, 다음과 같은 대화 데이터를 모델이 학습해야 한다고 가정합니다. 사용자가 안녕하세요, 오늘 날씨는 어떤가요?라고 물었고, 모델은 안녕하세요! 오늘 날씨는 맑고 화창합니다.라고 응답해야 합니다.

Gemma 4에서는 다음과 같은 템플릿 구조를 사용합니다:

```text
<|turn>system
You are a helpful assistant.<turn|>
<|turn>user
안녕하세요, 오늘 날씨는 어떤가요?<turn|>
<|turn>model
안녕하세요! 오늘 날씨는 맑고 화창합니다.<turn|>
```

여기서 데이터셋의 messages에는 `assistant` role로 들어가지만, Gemma 4의 chat template을 적용하면 모델 응답 턴은 `model` 턴으로 렌더링됩니다.

이 전체 텍스트는 토크나이저에 의해 정수 시퀀스로 변환됩니다. (해당 정수 시퀀스는 임의로 지정한 것으로 실제 정수와 다를 수 있습니다.)

```python
input_ids = [
    # <|turn>system\nYou are a helpful assistant.<turn|>\n
    1001, 1002, 13, 1003, 1004, 1005, 1006, 1007, 1008, 13,
    # <|turn>user\n안녕하세요, 오늘 날씨는 어떤가요?<turn|>\n
    2001, 2002, 13, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 13,
    # <|turn>model\n
    3001, 3002, 13,
    # 안녕하세요! 오늘 날씨는 맑고 화창합니다.<turn|>
    5001, 5002, 5003, 5004, 5005, 5006, 5007, 5008
]
```

모델이 예측해야 할 영역은 model의 최종 응답 부분입니다.

따라서 labels는 다음과 같이 설정됩니다:

```python
labels = [
    # system 부분 마스킹
    -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
    # user 부분 마스킹
    -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
    # <|turn>model\n 마스킹
    -100, -100, -100,
    # 안녕하세요! 오늘 날씨는 맑고 화창합니다.<turn|> (마스킹 없음)
    5001, 5002, 5003, 5004, 5005, 5006, 5007, 5008
]
```

이처럼 labels는 모델이 학습해야 할 model 응답 부분만을 포함하고, 나머지 부분은 -100으로 마스킹하여 손실 계산에서 제외됩니다. 모델은 `5001, 5002, 5003, 5004, 5005, 5006, 5007, 5008`에 해당하는 `안녕하세요! 오늘 날씨는 맑고 화창합니다.<turn|>` 전체 응답 영역을 생성하도록 학습할 수 있습니다.

현재 코드에서는 model 응답 시작 위치를 직접 문자열로 자르지 않고, 정답 응답을 제외한 messages에 `add_generation_prompt=True`를 적용하여 응답 시작 직전까지의 토큰 길이를 계산합니다.

```python
prompt_text = tokenizer.apply_chat_template(
    clean_messages[:-1],
    tokenize=False,
    add_generation_prompt=True
).strip()

prompt_tokenized = tokenizer(
    prompt_text,
    truncation=True,
    max_length=max_seq_length,
    padding=False,
    return_tensors=None,
)

assistant_start = len(prompt_tokenized["input_ids"])
```

그 다음 assistant_start 위치부터 실제 input_ids 값을 labels에 넣습니다.

```python
labels = [-100] * len(input_ids)

for j in range(assistant_start, len(input_ids)):
    labels[j] = input_ids[j]
```

결과적으로 system 부분, user 부분, `<|turn>model\n`에 해당하는 응답 시작 태그 부분은 -100으로 마스킹되고, 실제 model 응답 내용만 학습 대상이 됩니다.

# 5. 학습

In [17]:
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    data_collator=collate_fn,
    peft_config=peft_config,
)

In [18]:
# 학습 시작
trainer.train()   # 모델이 자동으로 허브와 output_dir에 저장됨

# 모델 저장
trainer.save_model()   # 최종 모델을 저장

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
10,0.826675
20,0.576681
30,0.490553
40,0.488186
50,0.438343
60,0.463448
70,0.415549
80,0.400335
90,0.444366
100,0.429905


# 6. 테스트 데이터 준비

In [19]:
prompt_lst = []
label_lst = []

for prompt in test_dataset["messages"]:
   input = tokenizer.apply_chat_template(
       prompt[:-1],
       tokenize=False,
       add_generation_prompt=True
   )
   label = prompt[-1]["content"]
   prompt_lst.append(input)
   label_lst.append(label)

In [20]:
print(prompt_lst[10])

<bos><|turn>system
당신은 주어진 뉴스로부터 종목에 영향을 주는 뉴스인지 판별하는 금융 뉴스 판별기입니다.
두 가지 답변 케이스가 존재하며 무조건 파이썬의 dictionary 형식으로 작성하십시오.
큰 따옴표 사이에 다른 따옴표들을 적으려고 시도하지 마십시오. 이는 dictionary 파싱을 실패하게 하는 원인이 됩니다. 따라서 주의하십시오.
아래 dictionary에서 각 value는 지시사항에 해당합니다. 지사사항을 따라 적지마십시오. 해당 지시사항에 따라 적절한 value를 채워넣으십시오.
해당사항이 없다면 빈 문자열 또는 빈 리스트로 적어야 합니다. 임의로 '없음' 등을 적어서는 안 됩니다.

만약 해당 뉴스가 특정 종목(회사)이 언급되지 않거나, 특정 종목(회사)와 아무런 연관이 없는 뉴스일 경우에는 아래와 같이 작성합니다.

답변:
{"is_stock_related": False,
"summary": "여기에는 해당 뉴스를 요약해서 요약문을 작성하십시오"}

만약 해당 뉴스가 특정 종목(회사)들과 연관되었거나, 특정 종목(회사)과 아무런 연관이 없는 뉴스일 경우에는 아래와 같이 작성합니다.

답변:
{"is_stock_related": True,
"positive_impact_stocks": ["파이썬 문자열 리스트의 형태로 이 뉴스가 긍정적인 영향을 줄것으로 추정되는 종목들의 이름을 작성하십시오. 약자로 적거나 별명으로 적지마십시오. 종목명으로 추정되는 한글명을 적으십시오. 뉴스로부터 추정할 수 있는 정확한 풀네임으로 적으십시오. 만약, 존재하지 않는다면 빈 리스트로 작성하십시오."],
"reason_for_positive_impact": "위의 종목들이 해당 뉴스로부터 긍정적인 영향을 받을 것으로 추정한 이유를 여기에다가 작성하십시오",
"positive_keywords": ["긍정적인 영향을 줄 것으로 추정되는 종목들이 존재했다면 여기에 긍정적인 영향을 주는데 근거가 되었던 주요한 명사 키워드들을 파이썬 문자열 리스트 형태로 작성

In [21]:
print(label_lst[10])

{'is_stock_related': True, 'negative_impact_stocks': ['애플', '엔비디아', 'TSMC', '텐센트', '바이트댄스'], 'negative_keywords': ['스마트폰 판매 감소', '인력 감축', '비용 절감', '경제 둔화'], 'positive_impact_stocks': [], 'positive_keywords': [], 'reason_for_negative_impact': '가트너의 보고서에 따르면 스마트폰 판매량의 감소는 애플과 같은 제조업체뿐만 아니라 엔비디아와 TSMC 같은 반도체 업체에게 부정적인 영향을 미칠 것으로 보입니다. 또한, 텐센트와 바이트댄스가 인력을 추가 감원할 계획을 발표함에 따라 이들의 주가에 부정적인 영향이 예상됩니다.', 'reason_for_positive_impact': '', 'summary': '가트너는 올해 전세계 스마트폰 판매량이 7% 감소할 것이라고 전망하며, 애플과 반도체 업체들이 영향을 받을 것으로 보입니다. 한편, 텐센트와 바이트댄스는 하반기 대규모 감원을 계획하고 있어, 글로벌 빅테크 기업들이 경제 둔화와 비용 절감 압박에 직면하고 있습니다.'}


# 7. 파인 튜닝 모델 테스트

`AutoPeftModelForCausalLM()`의 입력으로 LoRA Adapter가 저장된 체크포인트의 주소를 넣으면 LoRA Adapter가 기존의 LLM과 부착되어 로드됩니다. 이 과정은 LoRA Adapter의 가중치를 사전 학습된 언어 모델(LLM)에 통합하여 미세 조정된 모델을 완성하는 것을 의미합니다.

`peft_model_id` 변수는 미세 조정된 가중치가 저장된 체크포인트의 경로를 나타냅니다. `"gemma4-e4b-finance-new-summarizer/checkpoint-186"`는 LoRA Adapter 가중치가 저장된 위치로, 이 경로에서 해당 가중치를 불러옵니다. 실제 체크포인트 번호는 학습 과정에서 저장된 체크포인트 번호에 맞게 변경해야 합니다.

`fine_tuned_model`은 `AutoPeftModelForCausalLM.from_pretrained` 메서드를 통해 체크포인트를 로드하여 생성됩니다. 이 메서드는 LLM과 LoRA Adapter를 결합하고, 최적화된 설정으로 모델을 메모리에 로드합니다. `device_map="auto"` 옵션은 모델을 자동으로 GPU에 배치합니다.

`pipeline`은 Hugging Face의 고수준 유틸리티로, NLP 작업(예: 텍스트 생성, 번역, 요약 등)을 간단히 수행할 수 있게 해줍니다. 이 코드에서 사용된 `pipeline("text-generation")`은 텍스트 생성 작업을 수행하기 위한 파이프라인 객체를 생성합니다. 파이프라인은 내부적으로 모델과 토크나이저를 관리하여, 입력 텍스트를 토큰화하고, 모델을 통해 생성된 결과를 다시 디코딩하여 사람이 읽을 수 있는 텍스트로 변환합니다.

이 코드는 미세 조정된 LLM을 로드한 뒤, 이를 이용해 텍스트 생성 작업을 간단히 수행할 수 있도록 준비하는 데 목적이 있습니다. `pipeline`을 통해 텍스트 생성 작업을 실행하면, 입력 텍스트에 기반하여 모델이 다음 토큰을 예측하고 이를 반복적으로 생성합니다. 이 과정은 사용자에게 자연스러운 텍스트를 출력하는 데 사용됩니다.

In [22]:
import torch
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer, pipeline

In [23]:
peft_model_id = "gemma4-e4b-finance-new-summarizer/checkpoint-186"

fine_tuned_model = AutoPeftModelForCausalLM.from_pretrained(
    peft_model_id,
    device_map="auto",
    dtype=torch.bfloat16
)

pipe = pipeline("text-generation", model=fine_tuned_model, tokenizer=tokenizer)

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

In [24]:
eos_token = tokenizer("<turn|>", add_special_tokens=False)["input_ids"][0]

In [26]:
eos_token

106

In [25]:
def test_inference(pipe, prompt):
    outputs = pipe(prompt, max_new_tokens=1024, eos_token_id=eos_token, do_sample=False)
    return outputs[0]['generated_text'][len(prompt):].strip()

임의로 테스트 데이터 10번부터 14번까지 확인해봅시다.


In [27]:
for prompt, label in zip(prompt_lst[10:15], label_lst[10:15]):
    # print(f"    prompt:\n{prompt}")
    print(f"    response:\n{test_inference(pipe, prompt)}")
    print(f"    label:\n{label}")
    print("-"*50)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'eos_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_n

    response:
{'is_stock_related': True, 'negative_impact_stocks': ['애플', '엔비디아', 'TSMC'], 'negative_keywords': ['스마트폰 판매량 감소', '중국 봉쇄조치', '인플레이션', '수요 부진'], 'positive_impact_stocks': [], 'positive_keywords': [], 'reason_for_negative_impact': '글로벌 스마트폰 판매량 감소 전망으로 인해 애플, 엔비디아, TSMC와 같은 스마트폰 제조사 및 반도체 업체들이 압력을 받을 것으로 예상된다.', 'reason_for_positive_impact': '', 'summary': '글로벌 스마트폰 판매량이 7% 감소할 것으로 전망되며, 애플, 엔비디아, TSMC 등 스마트폰 제조사 및 반도체 업체들이 압력을 받을 것으로 예상된다. 또한, 유럽연합은 가상자산의 돈세탁을 막기 위한 규제에 합의했으며, 미국 저비용 항공사 스피릿 항공의 인수전이 뜨거워지고 있다. 텐센트와 바이트댄스 등 중국 빅테크 기업들은 경제 둔화로 인해 추가 감원을 준비하고 있다.'}<turn|>
    label:
{'is_stock_related': True, 'negative_impact_stocks': ['애플', '엔비디아', 'TSMC', '텐센트', '바이트댄스'], 'negative_keywords': ['스마트폰 판매 감소', '인력 감축', '비용 절감', '경제 둔화'], 'positive_impact_stocks': [], 'positive_keywords': [], 'reason_for_negative_impact': '가트너의 보고서에 따르면 스마트폰 판매량의 감소는 애플과 같은 제조업체뿐만 아니라 엔비디아와 TSMC 같은 반도체 업체에게 부정적인 영향을 미칠 것으로 보입니다. 또한, 텐센트와 바이트댄스가 인력을 추가 감원할 계획을 발표함에 따라 이들의 주가에 부정적인 영향이 예상됩니다.', '

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    response:
{'is_stock_related': True, 'negative_impact_stocks': [], 'negative_keywords': [], 'positive_impact_stocks': ['야놀자', '포커스미디어'], 'positive_keywords': ['야놀자', '포커스미디어', '동네가게 오래함께 캠페인', '광고 제작', '지역경제 활성화'], 'reason_for_negative_impact': '', 'reason_for_positive_impact': '야놀자와 포커스미디어가 동네가게 오래함께 캠페인을 통해 지역 소상공인을 위한 맞춤형 광고를 제작하고 홍보를 지원함으로써 지역경제 활성화에 기여할 것으로 기대된다. 이는 양사의 브랜드 이미지 제고와 매출 증대에 긍정적인 영향을 미칠 수 있다.', 'summary': '야놀자가 포커스미디어와 함께 동네가게 오래함께 캠페인을 진행하여 지역 소상공인을 위한 맞춤형 광고를 제작하고 홍보를 지원한다. 이 캠페인은 지역경제 활성화에 기여할 것으로 기대된다.'}<turn|>
    label:
{'is_stock_related': True, 'negative_impact_stocks': [], 'negative_keywords': [], 'positive_impact_stocks': ['야놀자'], 'positive_keywords': ['야놀자', '포커스미디어', '소상공인 지원', '광고 캠페인', '지역경제 활성화'], 'reason_for_negative_impact': '', 'reason_for_positive_impact': '야놀자가 포커스미디어와 협력하여 지역 소상공인을 지원하는 캠페인을 진행함으로써 지역사회와의 상생 및 경제 활성화에 기여하며, 이는 브랜드 이미지와 매출 성장에 긍정적인 영향을 미칠 수 있다.', 'summary': "야놀자가 포커스미디어와 함께 지역 소상공인을 지원하기 위한 '동네가게 오래함께' 캠페인을 진행하여 소상공인들의 인지도와 매출

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    response:
{'is_stock_related': True, 'negative_impact_stocks': [], 'negative_keywords': [], 'positive_impact_stocks': ['삼성바이오로직스'], 'positive_keywords': ['삼성바이오로직스', 'MSD', '위탁생산계약', '매출액'], 'reason_for_negative_impact': '', 'reason_for_positive_impact': '삼성바이오로직스가 미국 제약기업 MSD와 대규모 의약품 위탁생산 공급계약을 체결하여 매출액이 증가할 가능성이 높기 때문이다.', 'summary': '삼성바이오로직스가 미국 제약기업 MSD와 2768억2938만원 규모의 의약품 위탁생산 공급계약을 체결했다. 이 계약은 2022년 7월부터 2028년 12월까지 유효하며, 고객사의 수요 증가에 따라 계약금액이 증가할 수 있다.'}<turn|>
    label:
{'is_stock_related': True, 'negative_impact_stocks': [], 'negative_keywords': [], 'positive_impact_stocks': ['삼성바이오로직스'], 'positive_keywords': ['위탁생산계약', 'MSD', '매출 증대'], 'reason_for_negative_impact': '', 'reason_for_positive_impact': '삼성바이오로직스가 미국 제약기업인 MSD와 2768억원 규모의 위탁생산계약을 체결하였으므로, 이는 회사의 매출 증대에 긍정적 영향을 미칠 수 있다.', 'summary': '삼성바이오로직스가 미국 제약기업 MSD와 2768억원 규모의 의약품 위탁생산 계약을 체결하였으며, 이는 회사의 매출 대비 상당한 규모로, 향후 매출 증가가 기대된다.'}
--------------------------------------------------


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    response:
{'is_stock_related': True, 'negative_impact_stocks': [], 'negative_keywords': [], 'positive_impact_stocks': ['LG전자', 'SM엔터테인먼트'], 'positive_keywords': ['LG전자', 'SM엔터테인먼트', '피트니스 캔디', '디지털 피트니스', 'K POP', '메타버스'], 'reason_for_negative_impact': '', 'reason_for_positive_impact': 'LG전자와 SM엔터테인먼트가 합작하여 디지털 피트니스 콘텐츠 브랜드인 피트니스 캔디를 발표하면서, 두 회사의 기술력과 콘텐츠 경쟁력이 결합되어 새로운 시장을 개척할 가능성이 높아졌기 때문이다.', 'summary': 'LG전자와 SM엔터테인먼트가 디지털 피트니스 콘텐츠 합작 브랜드인 피트니스 캔디를 발표했다. 이 브랜드는 LG전자의 디지털 기술력과 SM엔터테인먼트의 K POP 콘텐츠를 결합하여 새로운 디지털 피트니스 라이프스타일을 제공하며, 글로벌 피트니스 및 헬스케어 산업에 기여할 것으로 기대된다.'}<turn|>
    label:
{'is_stock_related': True, 'negative_impact_stocks': [], 'negative_keywords': [], 'positive_impact_stocks': ['LG전자', 'SM엔터테인먼트'], 'positive_keywords': ['피트니스 캔디', '디지털 피트니스', 'LG전자', 'SM엔터테인먼트', 'K POP', '메타버스'], 'reason_for_negative_impact': '', 'reason_for_positive_impact': "LG전자와 SM엔터테인먼트가 공동으로 피트니스 콘텐츠 브랜드 '피트니스 캔디'를 출시하며 디지털 피트니스 시장에 진출하여, 양사의 디지털 기술력과 K POP 콘텐츠를 바탕으로 새로운 수익 창출과 시장 기회를 얻게 될 가능성이 높

# 8. 기본 모델 테스트

이번에는 LoRA Adapter를 merge하지 않은 기본 모델로 테스트 데이터에 대해서 인퍼런스해보겠습니다.

In [28]:
base_model_id = "google/gemma-4-E4B-it"

model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    dtype=torch.bfloat16
)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

In [29]:
for prompt, label in zip(prompt_lst[10:15], label_lst[10:15]):
    # print(f"    prompt:\n{prompt}")
    print(f"    response:\n{test_inference(pipe, prompt)}")
    print(f"    label:\n{label}")
    print("-"*50)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    response:
```json
{
"is_stock_related": true,
"positive_impact_stocks": [],
"reason_for_positive_impact": "",
"positive_keywords": [],
"negative_impact_stocks": [
"애플",
"엔비디아",
"TSMC",
"텐센트",
"바이트댄스"
],
"reason_for_negative_impact": "글로벌 스마트폰 판매량 감소 전망, 중국 경제 둔화 및 빅테크 기업들의 대규모 구조조정 계획 등이 언급되어 관련 기업들의 실적 및 투자 심리에 부정적인 영향을 미칠 수 있기 때문입니다.",
"negative_keywords": [
"스마트폰 판매량 감소",
"경제 전반에 걸친 침체 우려",
"중국의 봉쇄조치 여파",
"인플레이션",
"텐센트",
"바이트댄스",
"대규모 구조조정"
],
"summary": "글로벌 스마트폰 판매량 감소 전망과 더불어, EU의 가상자산 규제 움직임, 미국 항공사 인수전, 그리고 중국 빅테크 기업들의 추가적인 감원 계획 등 다양한 글로벌 경제 및 산업 동향을 다루고 있습니다."
}
```<turn|>
    label:
{'is_stock_related': True, 'negative_impact_stocks': ['애플', '엔비디아', 'TSMC', '텐센트', '바이트댄스'], 'negative_keywords': ['스마트폰 판매 감소', '인력 감축', '비용 절감', '경제 둔화'], 'positive_impact_stocks': [], 'positive_keywords': [], 'reason_for_negative_impact': '가트너의 보고서에 따르면 스마트폰 판매량의 감소는 애플과 같은 제조업체뿐만 아니라 엔비디아와 TSMC 같은 반도체 업체에게 부정적인 영향을 미칠 것으로 보입니다. 또한, 텐센트와 바이트댄스가 인력을 추가 감원할 계획을 발표함에 따라 이들의 주가에 부정적인 영향이 예상됩니다

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    response:
```json
{
"is_stock_related": true,
"positive_impact_stocks": [
"야놀자"
],
"reason_for_positive_impact": "야놀자가 포커스미디어와 협력하여 지역 소상공인을 위한 광고 캠페인을 진행하며, 이는 야놀자의 사회적 책임 활동 및 플랫폼 활용도를 높여 긍정적인 기업 이미지 제고 및 관련 사업 확장 기대감을 줄 수 있기 때문입니다.",
"positive_keywords": [
"야놀자",
"포커스미디어",
"동네가게 오래함께 캠페인",
"소상공인",
"광고 제작 및 송출 비용"
],
"negative_impact_stocks": [],
"reason_for_negative_impact": "",
"negative_keywords": [],
"summary": "야놀자가 포커스미디어와 협력하여 지역 내 우수 소상공인을 발굴하고 맞춤형 광고를 제작 및 송출하는 '동네가게 오래함께' 캠페인을 진행하며, 총 14억 원을 지원하여 지역 경제 활성화에 기여할 계획이다."
}
```<turn|>
    label:
{'is_stock_related': True, 'negative_impact_stocks': [], 'negative_keywords': [], 'positive_impact_stocks': ['야놀자'], 'positive_keywords': ['야놀자', '포커스미디어', '소상공인 지원', '광고 캠페인', '지역경제 활성화'], 'reason_for_negative_impact': '', 'reason_for_positive_impact': '야놀자가 포커스미디어와 협력하여 지역 소상공인을 지원하는 캠페인을 진행함으로써 지역사회와의 상생 및 경제 활성화에 기여하며, 이는 브랜드 이미지와 매출 성장에 긍정적인 영향을 미칠 수 있다.', 'summary': "야놀자가 포커스미디어와 함께 지역 소상공인을 지원하기 위한 '동네가게 오래함께' 캠페인을 진행하여 소상공인들

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    response:
{"is_stock_related": True, "positive_impact_stocks": ["삼성바이오로직스"], "reason_for_positive_impact": "삼성바이오로직스가 미국 제약기업 MSD와 2768억 2938만원 규모의 대규모 의약품 위탁생산 공급계약을 체결했기 때문에 매출 및 실적 개선 기대감이 높아졌기 때문입니다.", "positive_keywords": ["삼성바이오로직스", "MSD", "의약품 위탁생산 공급계약", "2768억2938만원"], "negative_impact_stocks": [], "reason_for_negative_impact": "", "negative_keywords": [], "summary": "삼성바이오로직스가 미국 제약기업 MSD와 2768억 2938만원 규모의 의약품 위탁생산 공급계약을 체결했으며, 이 계약은 2022년 7월부터 2028년 12월까지 유효합니다."}<turn|>
    label:
{'is_stock_related': True, 'negative_impact_stocks': [], 'negative_keywords': [], 'positive_impact_stocks': ['삼성바이오로직스'], 'positive_keywords': ['위탁생산계약', 'MSD', '매출 증대'], 'reason_for_negative_impact': '', 'reason_for_positive_impact': '삼성바이오로직스가 미국 제약기업인 MSD와 2768억원 규모의 위탁생산계약을 체결하였으므로, 이는 회사의 매출 증대에 긍정적 영향을 미칠 수 있다.', 'summary': '삼성바이오로직스가 미국 제약기업 MSD와 2768억원 규모의 의약품 위탁생산 계약을 체결하였으며, 이는 회사의 매출 대비 상당한 규모로, 향후 매출 증가가 기대된다.'}
--------------------------------------------------


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    response:
```json
{
"is_stock_related": true,
"positive_impact_stocks": [
"LG전자",
"SM엔터테인먼트"
],
"reason_for_positive_impact": "LG전자와 SM엔터테인먼트가 디지털 피트니스 콘텐츠 합작 브랜드 '피트니스 캔디'를 발표하며 새로운 사업 영역을 개척하고, 스마트 가전 및 콘텐츠 기술력을 결합하여 글로벌 피트니스 및 헬스케어 산업에 진출하려는 계획을 밝혔기 때문입니다.",
"positive_keywords": [
"피트니스 캔디",
"LG전자",
"SM엔터테인먼트",
"디지털 피트니스 콘텐츠",
"스마트 가전",
"K POP"
],
"negative_impact_stocks": [],
"reason_for_negative_impact": "",
"negative_keywords": [],
"summary": "LG전자와 SM엔터테인먼트가 디지털 피트니스 콘텐츠 합작 브랜드 '피트니스 캔디'를 발표했다. 이 브랜드는 LG전자의 디지털 기술력과 SM엔터테인먼트의 KPOP 콘텐츠를 결합하여 MZ세대를 겨냥한 맞춤형 모바일 피트니스 플랫폼을 제공하며, 글로벌 피트니스 및 헬스케어 시장 진출을 목표로 한다."
}
```<turn|>
    label:
{'is_stock_related': True, 'negative_impact_stocks': [], 'negative_keywords': [], 'positive_impact_stocks': ['LG전자', 'SM엔터테인먼트'], 'positive_keywords': ['피트니스 캔디', '디지털 피트니스', 'LG전자', 'SM엔터테인먼트', 'K POP', '메타버스'], 'reason_for_negative_impact': '', 'reason_for_positive_impact': "LG전자와 SM엔터테인먼트가 공동으로 피트니스 콘텐츠 브랜드 '피트니스 캔디'를 출시하며 디지털 피트니스 시장에 진출하여, 양사의 디